In [1]:
import copy
from itertools import combinations

def all_valuations(variables):
    for r in range(len(variables) + 1):
        for true_variables in combinations(variables, r):
            result = {x: False for x in variables}
            result.update({x: True for x in true_variables})
            yield result

class Formula:
    def __init__(self):
        self.components = []

    def interpret(self, valuation):
        pass

    def __repr__(self):
        return str(self)

    def __eq__(self, rhs): 
        return Eq(self.copy(), rhs.copy())

    def __and__(self, rhs): 
        return And(self.copy(), rhs.copy())

    def __or__(self, rhs):
        return Or(self.copy(), rhs.copy())

    def __rshift__(self, rhs):
        return Impl(self.copy(), rhs.copy())

    def __invert__(self):
        return Not(self.copy())
    
    def copy(self):
        return copy.deepcopy(self)

    def get_all_variables(self):
        result = set()
        for c in self.components:
            result.update(c.get_all_variables())
        return result

    def is_valid(self):
        variables = list(self.get_all_variables())
        for valuation in all_valuations(variables):
            if self.interpret(valuation) == False:
                return False, valuation
        return True, None

    def is_satisfiable(self):
        variables = list(self.get_all_variables())
        for valuation in all_valuations(variables):
            if self.interpret(valuation) == True:
                return True, valuation
        return False, None

    def is_contradictory(self):
        variables = list(self.get_all_variables())
        for valuation in all_valuations(variables):
            if self.interpret(valuation) == True:
                return False, valuation
        return True, None

    def is_falsifiable(self):
        variables = list(self.get_all_variables())
        for valuation in all_valuations(variables):
            if self.interpret(valuation) == False:
                return True, valuation
        return False, None

    def all_valuations_that_interpret_to_true(self):
        result = []
        variables = list(self.get_all_variables())
        for valuation in all_valuations(variables):
            if self.interpret(valuation) == True:
                result.append(valuation)
        return result

    
class Var(Formula):
    def __init__(self, name):
        super().__init__()
        self.name = name

    def interpret(self, valuation):
        return valuation[self.name]

    def get_all_variables(self):
        return set([self.name]) 

    def __str__(self):
        return self.name

class Const(Formula):
    def __init__(self, value):
        super().__init__()
        self.value = value
    
    def interpret(self, valuation):
        return self.value

    def __str__(self):
        return "{}".format(1 if self.value else 0)

class And(Formula):
    def __init__(self, lhs, rhs):
        super().__init__()
        self.components = [lhs, rhs]

    def interpret(self, valuation):
        return self.components[0].interpret(valuation) and self.components[1].interpret(valuation)

    def __str__(self):
        return f"({self.components[0]}) & ({self.components[1]})"


class Or(Formula):
    def __init__(self, lhs, rhs):
        super().__init__()
        self.components = [lhs, rhs]

    def interpret(self, valuation):
        return self.components[0].interpret(valuation) or self.components[1].interpret(valuation)

    def __str__(self):
        return f"({self.components[0]}) | ({self.components[1]})"


class Impl(Formula):
    def __init__(self, lhs, rhs):
        super().__init__()
        self.components = [lhs, rhs]

    def interpret(self, valuation):
        
        return not self.components[0].interpret(valuation) or self.components[1].interpret(valuation)

    def __str__(self):
        return f"({self.components[0]}) >> ({self.components[1]})"

class Eq(Formula):
    def __init__(self, lhs, rhs):
        super().__init__()
        self.components = [lhs, rhs]
    
    def interpret(self, valuation):
        return self.components[0].interpret(valuation) == self.components[1].interpret(valuation)

    def __str__(self):
        return f"({self.components[0]}) == ({self.components[1]})"

class Not(Formula):
    def __init__(self, op):
        super().__init__()
        self.components = [op]

    def interpret(self, valuation):
        return not self.components[0].interpret(valuation)

    def __str__(self):
        return f"~({self.components[0]})"

def vars(names):
    return [Var(name.strip()) for name in names.split(',')]

def evaluate_formula(formula):
    print(formula)
    print("is_valid: ", formula.is_valid())
    print("is_satisfiable: ", formula.is_satisfiable())
    print("is_falsifiable: ", formula.is_falsifiable())
    print("is_contradictory: ", formula.is_contradictory())
    print("all true valuations: ")
    for val in all_valuations(formula.get_all_variables()):
        if formula.interpret(val):
            print(val)

In [13]:
A,B,C = vars("A,B,C")
formula = (A | B) & ~(A & B) \
        & ~(~A & ~B & ~C) \
        & (A | B) \
        & (B | C) \
        & (A | C) \
        & ~(A & B & C)
evaluate_formula(formula)

(((((((A) | (B)) & (~((A) & (B)))) & (~(((~(A)) & (~(B))) & (~(C))))) & ((A) | (B))) & ((B) | (C))) & ((A) | (C))) & (~(((A) & (B)) & (C)))
is_valid:  (False, {'A': False, 'B': False, 'C': False})
is_satisfiable:  (True, {'A': True, 'B': False, 'C': True})
is_falsifiable:  (True, {'A': False, 'B': False, 'C': False})
is_contradictory:  (False, {'A': True, 'B': False, 'C': True})
all true valuations: 
{'A': True, 'B': False, 'C': True}
{'A': False, 'B': True, 'C': True}


In [15]:
A,B = vars("A,B")
formula = (A|B) & ~(A&B) & ~(~A & ~B)
evaluate_formula(formula)

(((A) | (B)) & (~((A) & (B)))) & (~((~(A)) & (~(B))))
is_valid:  (False, {'A': False, 'B': False})
is_satisfiable:  (True, {'A': True, 'B': False})
is_falsifiable:  (True, {'A': False, 'B': False})
is_contradictory:  (False, {'A': True, 'B': False})
all true valuations: 
{'A': True, 'B': False}
{'A': False, 'B': True}


In [17]:
A,B,C,D = vars("A,B,C,D")
formula = ((A|B) & ~(A & B) & ~(~A & ~B)) & ((C|D) & ~(C & D) & ~(~C & ~D))
evaluate_formula(formula)

((((A) | (B)) & (~((A) & (B)))) & (~((~(A)) & (~(B))))) & ((((C) | (D)) & (~((C) & (D)))) & (~((~(C)) & (~(D)))))
is_valid:  (False, {'A': False, 'B': False, 'C': False, 'D': False})
is_satisfiable:  (True, {'A': True, 'B': False, 'C': True, 'D': False})
is_falsifiable:  (True, {'A': False, 'B': False, 'C': False, 'D': False})
is_contradictory:  (False, {'A': True, 'B': False, 'C': True, 'D': False})
all true valuations: 
{'A': True, 'B': False, 'C': True, 'D': False}
{'A': True, 'B': False, 'C': False, 'D': True}
{'A': False, 'B': True, 'C': True, 'D': False}
{'A': False, 'B': True, 'C': False, 'D': True}


In [19]:
A,B,C = vars("A,B,C")
formula = (A==B) & (B==C)
evaluate_formula(formula)

((A) == (B)) & ((B) == (C))
is_valid:  (False, {'A': True, 'B': False, 'C': False})
is_satisfiable:  (True, {'A': False, 'B': False, 'C': False})
is_falsifiable:  (True, {'A': True, 'B': False, 'C': False})
is_contradictory:  (False, {'A': False, 'B': False, 'C': False})
all true valuations: 
{'A': False, 'B': False, 'C': False}
{'A': True, 'B': True, 'C': True}


In [20]:
A, B, C, D = vars("A,B,C,D")
formula = (B | D) & ~(B & D) & (A | C) & ~(A & C)
evaluate_formula(formula)

((((B) | (D)) & (~((B) & (D)))) & ((A) | (C))) & (~((A) & (C)))
is_valid:  (False, {'D': False, 'A': False, 'B': False, 'C': False})
is_satisfiable:  (True, {'D': True, 'A': True, 'B': False, 'C': False})
is_falsifiable:  (True, {'D': False, 'A': False, 'B': False, 'C': False})
is_contradictory:  (False, {'D': True, 'A': True, 'B': False, 'C': False})
all true valuations: 
{'D': True, 'A': True, 'B': False, 'C': False}
{'D': True, 'A': False, 'B': False, 'C': True}
{'D': False, 'A': True, 'B': True, 'C': False}
{'D': False, 'A': False, 'B': True, 'C': True}


In [21]:
A, B, C, D = vars("A,B,C,D")
formula = (A==D) & (B==C) & ~(A == B & B == C & C == D)
evaluate_formula(formula)

(((A) == (D)) & ((B) == (C))) & (~(((C) & (C)) == (D)))
is_valid:  (False, {'D': False, 'A': False, 'B': False, 'C': False})
is_satisfiable:  (True, {'D': True, 'A': True, 'B': False, 'C': False})
is_falsifiable:  (True, {'D': False, 'A': False, 'B': False, 'C': False})
is_contradictory:  (False, {'D': True, 'A': True, 'B': False, 'C': False})
all true valuations: 
{'D': True, 'A': True, 'B': False, 'C': False}
{'D': False, 'A': False, 'B': True, 'C': True}


In [22]:
A, B, C = vars("A,B,C") # Plavo - True, Crveno - False
formula = (~A >> (B==C)) | (~B >> C)
evaluate_formula(formula)

((~(A)) >> ((B) == (C))) | ((~(B)) >> (C))
is_valid:  (True, None)
is_satisfiable:  (True, {'A': False, 'B': False, 'C': False})
is_falsifiable:  (False, None)
is_contradictory:  (False, {'A': False, 'B': False, 'C': False})
all true valuations: 
{'A': False, 'B': False, 'C': False}
{'A': True, 'B': False, 'C': False}
{'A': False, 'B': True, 'C': False}
{'A': False, 'B': False, 'C': True}
{'A': True, 'B': True, 'C': False}
{'A': True, 'B': False, 'C': True}
{'A': False, 'B': True, 'C': True}
{'A': True, 'B': True, 'C': True}


In [24]:
A, B, C = vars("A,B,C")
formula = ~(A==B) & ~(A==C) & ~(B==C)
evaluate_formula(formula)

((~((A) == (B))) & (~((A) == (C)))) & (~((B) == (C)))
is_valid:  (False, {'A': False, 'B': False, 'C': False})
is_satisfiable:  (False, None)
is_falsifiable:  (True, {'A': False, 'B': False, 'C': False})
is_contradictory:  (True, None)
all true valuations: 


In [26]:
A,B,C,D = vars("A,B,C,D")
formula = (A >> (~B | ~C | ~D)) & (~D >> ((A & B) | (B & C) | (A & C))) \
        & ~((A == B) & (B == C) & (C == D))
evaluate_formula(formula)

(((A) >> (((~(B)) | (~(C))) | (~(D)))) & ((~(D)) >> ((((A) & (B)) | ((B) & (C))) | ((A) & (C))))) & (~((((A) == (B)) & ((B) == (C))) & ((C) == (D))))
is_valid:  (False, {'D': False, 'B': False, 'A': False, 'C': False})
is_satisfiable:  (True, {'D': True, 'B': False, 'A': False, 'C': False})
is_falsifiable:  (True, {'D': False, 'B': False, 'A': False, 'C': False})
is_contradictory:  (False, {'D': True, 'B': False, 'A': False, 'C': False})
all true valuations: 
{'D': True, 'B': False, 'A': False, 'C': False}
{'D': True, 'B': True, 'A': False, 'C': False}
{'D': True, 'B': False, 'A': True, 'C': False}
{'D': True, 'B': False, 'A': False, 'C': True}
{'D': False, 'B': True, 'A': True, 'C': False}
{'D': False, 'B': True, 'A': False, 'C': True}
{'D': False, 'B': False, 'A': True, 'C': True}
{'D': True, 'B': True, 'A': True, 'C': False}
{'D': True, 'B': True, 'A': False, 'C': True}
{'D': True, 'B': False, 'A': True, 'C': True}
{'D': False, 'B': True, 'A': True, 'C': True}
